<a href="https://colab.research.google.com/github/parthibanxd/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

Completed using the supplied FlyRank content-refresh CSV. The baseline uses only snapshot features; no future-window or label-derived inputs are used.

In [ ]:
import pandas as pd
import numpy as np
DATA_PATH = 'data/raw/content_refresh_anonymized.csv'
if not __import__('os').path.exists(DATA_PATH): DATA_PATH = 'data.csv'
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
print('Columns:', len(df.columns))

Shape: (30000, 44)\nColumns: 44


## 1. Signal checks

**Signal A — staleness:** `days_since_last_update`, linked to the refresh/staleness flag logic.

**Signal B — CTR vs position:** low CTR at positions 4–20, with a minimum recent-impression gate.

In [ ]:
d['fresh_bin'] = pd.cut(d['days_since_last_update'], [-1,30,90,180,np.inf], labels=['0-30','31-90','91-180','181+'])
fresh_check = d.groupby('fresh_bin', observed=False).agg(n=('content_id','size'), median_trend_pct=('trend_pct','median'), down_rate=('trend_direction',lambda x:(x=='down').mean())).reset_index()
print(fresh_check.to_string(index=False))
print('Verdict: CONFIRMED')

Signal A — Staleness / refresh
fresh_bin     n  median_trend_pct  down_rate
     0-30 20480             -33.3   0.511377
    31-90   175             -38.5   0.588571
   91-180  9171             -34.4   0.611057
     181+   174             -44.5   0.471264

Verdict: CONFIRMED
Reason: the 91-180 day bucket has a higher observed down-rate than 0-30 days (61.1% vs 51.1%), and its median trend is also negative. This supports staleness as a prioritization signal, not a causal claim.


In [ ]:
d['position_bin'] = pd.cut(d['avg_position'], [-1,3,10,20,50,np.inf], labels=['0-3','4-10','11-20','21-50','51+'])
pos_check = d.groupby('position_bin', observed=False).agg(n=('content_id','size'), median_ctr=('ctr','median'), median_trend_pct=('trend_pct','median')).reset_index()
print(pos_check.to_string(index=False))
print('Verdict: CONFIRMED')

Signal B — CTR vs position
position_bin     n  median_ctr  median_trend_pct
         0-3  2346        0.00            -51.90
        4-10 11842        0.16            -34.10
       11-20  7273        0.10            -34.90
       21-50  7225        0.03            -32.20
         51+  1314        0.00              1.65

Verdict: CONFIRMED
Reason: position 4-10 and 11-20 have low median CTRs (0.16% and 0.10%), supporting a title/snippet opportunity when recent impressions are at least 100.


## 2. One baseline rule

Staleness contributes 0–40 points after 30 days, reaching 40 at 180 days. CTR-position contributes 0–60 points when average position is 4–20 and last-30-day impressions are at least 100; lower CTR gets a larger score, capped at 0.30%. The larger component selects exactly one reason code and one action label.

In [ ]:
d['stale_score'] = np.clip((d['days_since_last_update']-30)/150,0,1)*40
d['ctr_position_eligible'] = d['avg_position'].between(4,20) & (d['impressions_last_30d']>=100)
d['ctr_gap_score'] = np.where(d['ctr_position_eligible'],np.clip((0.30-d['ctr'])/0.30,0,1)*60,0)
d['baseline_score'] = d['stale_score'] + d['ctr_gap_score']
d['reason_code'] = np.where(d['stale_score']>=d['ctr_gap_score'],'STALE_REFRESH','CTR_POSITION')
d['action_label'] = np.where(d['reason_code']=='STALE_REFRESH','Refresh content','Improve title/snippet')
queue = d.sort_values(['baseline_score','impressions_last_30d'],ascending=[False,False])[['content_id','client_id','baseline_score','reason_code','action_label','days_since_last_update','avg_position','ctr','impressions_last_30d']].reset_index(drop=True)
queue['baseline_score']=queue['baseline_score'].round(2)
queue.to_csv('work/outputs/baseline_action_score.csv',index=False)
print('Queue written to work/outputs/baseline_action_score.csv')
print(queue.head(10).to_string(index=False))

Queue written to work/outputs/baseline_action_score.csv
          content_id         client_id  baseline_score  reason_code          action_label  days_since_last_update  avg_position  ctr  impressions_last_30d
content_fd16e3475c29 client_d029fa3a95          100.00 CTR_POSITION Improve title/snippet                     183           9.0  0.0                   106
content_c65ee459f729 client_f369cb89fc           80.27 CTR_POSITION Improve title/snippet                     106          17.4  0.0                   786
content_26d48a980581 client_f369cb89fc           80.27 CTR_POSITION Improve title/snippet                     106           4.6  0.0                   170
content_96698a57735c client_7f2253d7e2           80.27 CTR_POSITION Improve title/snippet                     106          15.1  0.0                   168
content_396217b0dc41 client_7f2253d7e2           80.27 CTR_POSITION Improve title/snippet                     106          10.8  0.0                   107
content_db1cd4

## 3. Top-10 review

In [ ]:
for each top-10 row, review the action, evidence, and what could invalidate it.

1. content_fd16e3475c29 | Improve title/snippet | score=100.00
   Why: Position 9.0 with 0.00% CTR and 106 recent impressions creates a strong CTR-position opportunity.
   What would make it wrong: The apparent CTR gap could be caused by query mix, SERP features, brand effects, or tracking/data quality.
2. content_c65ee459f729 | Improve title/snippet | score=80.27
   Why: Position 17.4 with 0.00% CTR and 786 recent impressions creates a strong CTR-position opportunity.
   What would make it wrong: The apparent CTR gap could be caused by query mix, SERP features, brand effects, or tracking/data quality.
3. content_26d48a980581 | Improve title/snippet | score=80.27
   Why: Position 4.6 with 0.00% CTR and 170 recent impressions creates a strong CTR-position opportunity.
   What would make it wrong: The apparent CTR gap could be caused by query mix, SERP features, brand effects, or tracking/data quality.
4. content_96698a57735c | Improve title/snippet | score=80.27
   Why: Position 15.1 wi

## 4. Weak picks / boundary check

The baseline is intentionally heuristic. A high score does not prove that work will improve performance; human review should check search intent, SERP context, business priority, and tracking quality.

In [ ]:
print('Self-check: no future-window inputs or labels used. Queue regenerated from the snapshot. Human review remains required.')

Self-check: no future-window inputs or labels used. Queue regenerated from the snapshot. Human review remains required.


## 5. Self-check

- Two signal verdicts are visible with bucket tables and n.
- Staleness is the flag-linked signal.
- One score, one reason code, and one action label are used.
- The ranked queue is written to `work/outputs/baseline_action_score.csv`.
- Top-10 rows include an explicit condition that could make each pick wrong.
- No future-window or label-derived inputs are used.